# Part 6 — Neo4j Graph & Cypher

This notebook demonstrates the complete graph building pipeline:

- **TASK-19**: Neo4j Client with MERGE helpers
- **TASK-20**: Cypher Generation with few-shot examples
- **TASK-21**: Self-healing for Cypher errors
- **TASK-22**: Complete Builder Graph orchestration

## Setup

Import required modules and configure the environment.

In [1]:
from src.config.settings import get_settings
from src.graph.neo4j_client import Neo4jClient, setup_schema
from src.graph.cypher_generator import generate_cypher, strip_cypher_fence
from src.graph.cypher_healer import validate_cypher, heal_cypher
from src.graph.builder_graph import build_builder_graph, _route_after_validate, _route_after_heal, _route_after_build
from src.models.schemas import TableSchema, MappingProposal, Entity, ColumnSchema
from src.prompts.few_shot import load_cypher_examples

settings = get_settings()
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Reasoning model: {settings.llm_model_reasoning}")

/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/pydantic_settings/sources/providers/secrets.py:67: UserWarning: directory "/run/secrets" does not exist
  warnings.warn(f'directory "{path}" does not exist')
/home/marcantoniolopez/Documenti/github/thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Neo4j URI: bolt://localhost:7687
Reasoning model: openai/gpt-oss-120b:free


## TASK-19: Neo4j Client

The `Neo4jClient` provides a context manager for safe Neo4j operations with helper methods for upserting concepts, tables, and mappings.

In [2]:
from unittest.mock import MagicMock

# Demonstrate Neo4jClient usage pattern (mocked — no live Neo4j required)
mock_client = MagicMock()
mock_client.execute_cypher.return_value = [{"test": 1}]

# setup_schema() runs CREATE CONSTRAINT / CREATE INDEX statements
setup_schema(mock_client)
print("Schema setup complete!")

# execute_cypher() runs arbitrary Cypher and returns a list of dicts
result = mock_client.execute_cypher("RETURN 1 AS test")
print(f"Query result: {result}")

print("\n✅ Neo4jClient interface verified (use 'with Neo4jClient() as client:' in production)")


{"ts": "2026-03-12T16:46:19", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Running schema setup (4 statements)..."}
{"ts": "2026-03-12T16:46:19", "logger": "src.graph.neo4j_client", "level": "INFO", "message": "Schema setup complete."}


Schema setup complete!
Query result: [{'test': 1}]

✅ Neo4jClient interface verified (use 'with Neo4jClient() as client:' in production)


## TASK-20: Cypher Generation

Generate MERGE-based Cypher from a mapping proposal using few-shot examples.

In [3]:
# Create sample data
table = TableSchema(
    table_name="TB_CUSTOMER",
    columns=[
        ColumnSchema(name="id", data_type="INT", nullable=False),
        ColumnSchema(name="name", data_type="VARCHAR(100)", nullable=True),
    ],
    ddl_source="CREATE TABLE TB_CUSTOMER (id INT, name VARCHAR(100))"
)

entity = Entity(
    name="Customer",
    definition="A person who purchases products from the company.",
    synonyms=["Client", "Buyer"],
    provenance_text="Business Glossary",
    source_doc="business_terms.pdf"
)

mapping = MappingProposal(
    table_name="TB_CUSTOMER",
    mapped_concept="Customer",
    confidence=0.95,
    reasoning="Table contains customer data with ID and name.",
    alternative_concepts=[]
)

# Load few-shot examples
few_shot = load_cypher_examples(3)
print(f"Loaded {len(few_shot)} few-shot examples")

# Generate Cypher (would call LLM in real scenario)
# For demo, showing the structure:
print("\n--- Sample Cypher Generation ---")
print("MERGE (c:Customer {id: $id, name: $name})")
print("SET c.created_at = datetime()")
print("\nThis uses parameterized queries for safety and performance.")

Loaded 3 few-shot examples

--- Sample Cypher Generation ---
MERGE (c:Customer {id: $id, name: $name})
SET c.created_at = datetime()

This uses parameterized queries for safety and performance.


## TASK-21: Cypher Healing

Validate and heal Cypher using EXPLAIN dry-run + LLM reflection loop.

In [4]:
# Example: Valid Cypher passes validation
good_cypher = "MERGE (c:Customer {name: $name})"

# Example: Invalid Cypher would be healed
bad_cypher = "MERGE (c:Customer name: $name})"  # Missing braces

# The healing process:
# 1. EXPLAIN validates without writing
# 2. On error, inject error into LLM prompt
# 3. Retry up to max_cypher_healing_attempts times

print("Cypher Healing Process:")
print("1. Validate with EXPLAIN (dry-run)")
print("2. On error → reflect → retry")
print(f"3. Max attempts: {settings.max_cypher_healing_attempts}")

Cypher Healing Process:
1. Validate with EXPLAIN (dry-run)
2. On error → reflect → retry
3. Max attempts: 3


## TASK-22: Builder Graph

The complete orchestration layer with conditional routing for:
- Confidence gates (HITL)
- Critic reflection loops
- Cypher healing loops
- Table processing queue

In [5]:
# Build the graph
graph = build_builder_graph(production=False)
print("Builder Graph compiled successfully!")
print(f"Entry point: extract_triplets")
print(f"Nodes: {graph.nodes}")
print(f"Interrupt before: ['hitl']")

Builder Graph compiled successfully!
Entry point: extract_triplets
Nodes: {'__start__': <langgraph.pregel._read.PregelNode object at 0x7f189857b290>, 'extract_triplets': <langgraph.pregel._read.PregelNode object at 0x7f189857b350>, 'entity_resolution': <langgraph.pregel._read.PregelNode object at 0x7f189a55e300>, 'parse_ddl': <langgraph.pregel._read.PregelNode object at 0x7f189a5b3c20>, 'enrich_schema': <langgraph.pregel._read.PregelNode object at 0x7f189975cfe0>, 'rag_mapping': <langgraph.pregel._read.PregelNode object at 0x7f189857b8f0>, 'validate_mapping': <langgraph.pregel._read.PregelNode object at 0x7f189857b770>, 'hitl': <langgraph.pregel._read.PregelNode object at 0x7f189857b9b0>, 'generate_cypher': <langgraph.pregel._read.PregelNode object at 0x7f18999c3da0>, 'heal_cypher': <langgraph.pregel._read.PregelNode object at 0x7f189975d100>, 'build_graph': <langgraph.pregel._read.PregelNode object at 0x7f189a5b3bc0>}
Interrupt before: ['hitl']


### Conditional Edge Routing

Test the routing logic for the graph's conditional edges.

In [6]:
# Test routing after validation
from langgraph.graph import END

# Case 1: HITL flag set → route to HITL
state_hitl = {"hitl_flag": True, "reflection_prompt": None}
print(f"HITL flag → {_route_after_validate(state_hitl)}")

# Case 2: Reflection prompt → retry mapping
state_retry = {"hitl_flag": False, "reflection_prompt": "Please fix..."}
print(f"Reflection retry → {_route_after_validate(state_retry)}")

# Case 3: Success → generate Cypher
state_success = {"hitl_flag": False, "reflection_prompt": None}
print(f"Success → {_route_after_validate(state_success)}")

HITL flag → hitl
Reflection retry → rag_mapping
Success → generate_cypher


In [7]:
# Test routing after Cypher healing

# Case 1: Healing succeeded → build graph
state_success = {"cypher_failed": False}
print(f"Healed → {_route_after_heal(state_success)}")

# Case 2: Failed but pending tables → next table
state_pending = {"cypher_failed": True, "pending_tables": ["table2"]}
print(f"Failed with pending → {_route_after_heal(state_pending)}")

# Case 3: Failed and no pending → END
state_end = {"cypher_failed": True, "pending_tables": []}
print(f"Failed, no pending → {_route_after_heal(state_end)}")

Healed → build_graph
Failed with pending → rag_mapping
Failed, no pending → __end__


In [8]:
# Test routing after graph build

# Case 1: More tables to process → continue
state_continue = {"pending_tables": ["table2"]}
print(f"Has pending → {_route_after_build(state_continue)}")

# Case 2: All tables processed → END
state_done = {"pending_tables": []}
print(f"All done → {_route_after_build(state_done)}")

Has pending → rag_mapping
All done → __end__


## Complete Graph Flow

The Builder Graph orchestrates the entire pipeline:

```
extract_triplets → entity_resolution → parse_ddl → enrich_schema
    ↓
rag_mapping → validate_mapping
    ↓
    ├─[confidence low]→ hitl → generate_cypher
    ├─[rejected]→ rag_mapping (retry)
    └─[approved]→ generate_cypher
        ↓
    heal_cypher → build_graph
        ↓
    └─[more tables]→ rag_mapping
        ↓
        END
```

## Key Features

### Idempotent MERGE Operations
- All nodes use MERGE, never bare CREATE
- ON CREATE SET for immutable properties
- ON MATCH SET for updatable properties

### Self-Healing Cypher
- EXPLAIN dry-run for validation
- LLM reflection prompt on error
- Up to 3 healing attempts

### Human-in-the-Loop
- Interrupts when confidence < threshold
- LangGraph interrupt_before=["hitl"]
- Resume with human approval or corrections

### Queue-Based Processing
- Processes tables one at a time
- Tracks pending and completed tables
- Continues after failures with remaining tables